In [1]:
from datetime import datetime
from pathlib import Path
import calendar
import pandas as pd
import pikepdf

In [2]:
df = pd.read_csv('test.csv')
df.to_json('test.json', orient='records', indent=2)
df = pd.read_json('test.json')
data = df.to_dict(orient='records')
df

,prefix,password,company_code,company_name,account_no,bank_code,bank_name,bank_email
0,1500010331,E#199801002549,ERM,Erumas Sdn Bhd,100750010015955,ABMB,Alliance Bank Malaysia Berhad,e-statement@alliancebank.e-statement.com.my
1,2500029697,O#199901006813,OCM,Ocamat Sdn Bhd,100750010034721,ABMB,Alliance Bank Malaysia Berhad,e-statement@alliancebank.e-statement.com.my
2,3000090078,T#199701035240,TJ,Talu Jadi Sdn Bhd,100750010018277,ABMB,Alliance Bank Malaysia Berhad,e-statement@alliancebank.e-statement.com.my
3,3700032130,T#201101026242,TJP,Ocamat Plantations Sdn Bhd,600700013002795,ABMB,Alliance Bank Malaysia Berhad,e-statement@alliancebank.e-statement.com.my
4,ca_con_3712,675W980305,ERM,Erumas Sdn Bhd,21005500003712,RHB,RHB Bank Berhad,rhb@stmts.rhbgroup.com
5,ca_con_5542,740W971021,TJ,Talu Jadi Sdn Bhd,21005500055542,RHB,RHB Bank Berhad,rhb@stmts.rhbgroup.com
6,ca_con_7600,713X990422,OCM,Ocamat Sdn Bhd,21005500057600,RHB,RHB Bank Berhad,rhb@stmts.rhbgroup.com
7,ca_con_9390,378X110726,TJP,Talu Jadi Plantations Sdn Bhd,21005500089390,RHB,RHB Bank Berhad,rhb@stmts.rhbgroup.com


In [3]:
input_path = Path.cwd() / 'input'
output_path = Path.cwd() / 'output'
file_list = list(input_path.glob('*.pdf'))

In [4]:
for file_path in file_list:
    for item in data:
        prefix = item['prefix']
        if prefix in file_path.name:
            break
    prefix = item['prefix']
    password = item['password']
    company_code = item['company_code']
    bank_code = item['bank_code']
    if bank_code == 'RHB':
        date_string = file_path.name.split('_')[3]
        input_format = '%Y%m%d'
        output_format = '%Y-%m-%d'
        dt = datetime.strptime(date_string, input_format)
        date_string = dt.strftime(output_format)
    elif bank_code == 'ABMB':
        date_string = file_path.stem.replace(prefix, '')
        input_format = '%b%Y'
        output_format = '%Y-%m-%d'
        dt = datetime.strptime(date_string, input_format)
        dt_day = calendar.monthrange(dt.year, dt.month)[1]
        dt = dt.replace(day=dt_day)
        date_string = dt.strftime(output_format)
    else:
        pass
    target_path = output_path / f'{company_code}_{bank_code}_{date_string}.pdf'
    with pikepdf.open(file_path, password=password) as pdf:
        pdf.save(target_path)